## 1. Configurations

### 1-1. Import libraries

In [1]:
from    typing      import  Callable, Hashable, Sequence, Union
from    pathlib     import  Path
import  yaml
import  pickle

from    time            import  time
from    tqdm.notebook   import  trange, tqdm

import  numpy       as      np
import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  pandas      as      pd

import  append_path
from    deep_numerical              import  utils
from    deep_numerical.utils        import  GaussianNormalizer, relative_error
from    deep_numerical              import  numerical   as  dn
from    deep_numerical.numerical    import  distribution    as  D
from    train_utils                 import  load_data, augment_data_2D

from    model_keys      import  *
from    helper_2D       import  *
from    plots           import  *
DIMENSION = 2

dtype  = torch.float
device = torch.device("cuda:1")
BATCH_SIZE      = 1000
METRIC_ORDER    = 2.0

DO_AUGMENTATION = False

### 1-2. Load trained models and training histories

In [2]:
model_data      = load_model_data(f"paths_{DIMENSION}D.yaml")
models          = instantiate_models(model_data, device)
normalizers     = instantiate_normalizers(model_data)
train_histories = instantiate_histories(model_data)

[fno_param] The model is not included in the comparison.
[pfno_param] The model is not included in the comparison.
[prfno_param] The model is not included in the comparison.
[fno_kernel] The model is not included in the comparison.
[pfno_kernel] The model is not included in the comparison.
[prfno_kernel] The model is not included in the comparison.
[rfno_kernel] The model is not included in the comparison.


### 1-3. Plot of training data

Print of the number of paramters in each model

In [3]:
n_params = {
    'Models': [], 'Parameters': [],
}
for k, v in models.items():
    n_params['Models'].append(k)
    n_params['Parameters'].append(v.count_parameters())
n_params = pd.DataFrame(n_params)
n_params.set_index('Models', inplace=True)
print(n_params)

            Parameters
Models                
rfno_param      574625


## 2. Conduct test

In [4]:
# Test data
with open(f"config_test_{DIMENSION}D__one_inst.yaml") as f:
    test_config = yaml.load(f, Loader=yaml.FullLoader)

_data = test_config['pde_dataset']
TEST_PATH       = Path(_data['path_test'])
TEST_RESOLUTION = RESOLUTION = _data['resolution']
EXP_SPEED       = _data['exp_speed']
TEST_PART       = _data['test_part']
NUM_TIME_STEPS  = _data['num_time_steps']
arr_t = 0.1 * torch.arange(NUM_TIME_STEPS)

In [5]:
test_raw_data: dict[str, torch.Tensor] = {}

try:
    _test_data, _test_info = load_data(TEST_PATH, TEST_RESOLUTION, EXP_SPEED, TEST_PART, size_zfill=2)
except FileNotFoundError:
    _test_data, _test_info = load_data(TEST_PATH, TEST_RESOLUTION, EXP_SPEED, TEST_PART, size_zfill=3)
V_MAX           = _test_info['v_max']
V_EXTENT        = (-V_MAX, V_MAX, V_MAX, -V_MAX)
WHERE_CLOSED    = _test_info['v_where_closed']

In [6]:
_test_data['data'].device

device(type='cpu')

In [7]:
test_raw_data['data']   = _test_data['data'][[0], :NUM_TIME_STEPS]
test_raw_data['params'] = _test_data['vhs_alpha'][[0]]
test_gen_time = _test_info['elapsed_time']
if DO_AUGMENTATION:
    test_raw_data = augment_data_2D(test_raw_data)
    test_gen_time *= 8
TEST_SIZE   = test_raw_data['data'].shape[0]
del(_test_data, _test_info)

print(f"The size of the test dataset >>>", TEST_SIZE, sep=' ')
print(f"The shape of the test dataset >>>", test_raw_data['data'].shape, test_raw_data['params'].shape, sep=' ')
print(f"Time for generateing the test dataset >>> {test_gen_time:.2f} seconds")

test_arg_params:    dict[int, torch.LongTensor] = {}
params_unique = list(test_raw_data['params'].unique())
params_unique.sort()
params_unique = tuple(params_unique)
for idx, p in enumerate(params_unique):
    test_arg_params[idx] = torch.argwhere(test_raw_data['params'].flatten()==p).flatten()

The size of the test dataset >>> 1
The shape of the test dataset >>> torch.Size([1, 51, 128, 128, 1]) torch.Size([1, 1])
Time for generateing the test dataset >>> 21.53 seconds


In [8]:
print(f"Construct a test dataset for each model.")
test_datasets: dict[str, TensorDataset] = {
    k: construct_datasets(
        k,
        test_raw_data['data'], test_raw_data['params'], normalizers[k],
        V_MAX, res_params=10,
    ) for k in models.keys()
}

print(f"Construct a test dataloader for each model.")
test_dataloaders: dict[str, DataLoader] = {
    k: DataLoader(v, batch_size=BATCH_SIZE, shuffle=False)
    for k, v in test_datasets.items()
}

Construct a test dataset for each model.
Construct a test dataloader for each model.


In [9]:
print("Test the models")
print("* Initialize the storages of the predictions and the relative errors")
_TensorSet = Union[torch.Tensor, list[torch.Tensor]]
pred_lists:     dict[str, _TensorSet] = {k: [] for k in models.keys()}
"""Saves *decoded* predictions."""
rel_errors:     dict[str, _TensorSet] = {k: [] for k in models.keys()}
"""Saves the relative errors between the *decoded* predictions and targets."""
inference_times:    dict[str, float] = {k: 0.0 for k in models.keys()}


print("* Start evaluating the models")
for k_model, model in models.items():
    print(f"  - Evaluating [{k_model}]...")
    model.eval()
    normalizers[k_model]['data'].to(device)
    normalizers[k_model]['params'].to(device)
    inference_times[k_model] = time()

    with torch.no_grad():
        traj:   torch.Tensor
        params: torch.Tensor
        model_preds = []
        for traj, params in test_dataloaders[k_model]:
            # Move tensors to the device
            traj    = traj.to(device)
            params  = params.to(device)
            pred    = traj[:, 0]
            # Instantiate a storage for batched predictions
            temp_batch_preds = [
                normalizers[k_model]['data'].decode(traj[:, 0]).cpu()
            ]
            # Compute
            for idx in trange(NUM_TIME_STEPS - 1):
                # Time-marching (in the scaled space) and decoding
                pred = model.forward(pred, params)
                temp_batch_preds.append( normalizers[k_model]['data'].decode(pred).cpu() )
            # Stack predictions for the current batch
            model_preds.append( torch.stack(temp_batch_preds, dim=1) )

    inference_times[k_model] = time() - inference_times[k_model]
    pred_lists[k_model]      = torch.cat(model_preds, dim=0)
    rel_errors[k_model]      = relative_error(pred_lists[k_model], test_raw_data['data'], dim=tuple(range(2, 3+DIMENSION)))

print("Done.")

Test the models
* Initialize the storages of the predictions and the relative errors
* Start evaluating the models
  - Evaluating [rfno_param]...


  0%|          | 0/50 [00:00<?, ?it/s]

Done.


In [10]:
train_time = {
    k: v['train_time']
    for k, v in train_histories.items()
}
train_time_per_epoch = {
    k: v['train_time'] / len(v['train_rel_error'])
    for k, v in train_histories.items()
}
__elapsed_times = {
    'Models': [],
    'Training time': [],
    'Training time per epoch': [],
    'Inference time': [],
}
for k in models.keys():
    __elapsed_times['Models'].append(k)
    __elapsed_times['Training time'].append(train_time[k])
    __elapsed_times['Training time per epoch'].append(train_time_per_epoch[k])
    __elapsed_times['Inference time'].append(inference_times[k])
elapsed_times = pd.DataFrame(__elapsed_times)
elapsed_times.set_index('Models', inplace=True)
print(elapsed_times)

            Training time  Training time per epoch  Inference time
Models                                                            
rfno_param   39928.133082                39.928133        0.228978


In [11]:
for k in models.keys():
    print(f"[ {k} ]")
    print(f"- Inference time: {inference_times[k]:.2f} seconds")
    print(f"- Relative error at the terminal time (average, std_dev): {rel_errors[k][:, -1].mean().item():.2e}, {rel_errors[k][:, -1].std().item():.2e}")
    print(f"- Maximum relative error at the terminal time: {rel_errors[k][:, -1].max().item():.2e}")
    print(f"- Maximum relative error during inference: {rel_errors[k].max().item():.2e}")
    print('-' * 50)

[ rfno_param ]
- Inference time: 0.23 seconds
- Relative error at the terminal time (average, std_dev): 7.32e-03, nan
- Maximum relative error at the terminal time: 7.32e-03
- Maximum relative error during inference: 7.32e-03
--------------------------------------------------


/tmp/ipykernel_954604/788645031.py:4: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1808.)
  print(f"- Relative error at the terminal time (average, std_dev): {rel_errors[k][:, -1].mean().item():.2e}, {rel_errors[k][:, -1].std().item():.2e}")


End of file